# Advanced Problems with Solutions: Python Modules, Namespaces, and Import Internals

This notebook contains advanced, runnable practice problems about Python modules as objects, module namespaces, `sys.modules`, import caching, dynamic module creation, and safe import-state testing.

Best-practice themes used throughout:
- isolate experiments in temporary directories;
- restore `sys.path` and `sys.modules`;
- prefer assertions over manual inspection;
- avoid leaving global import state mutated;
- distinguish name bindings from module objects.

## Problem 1 — Prove that modules are cached objects

Create a temporary module named `demo_mod.py`, import it, delete only your local name binding, and import it again. Prove that the second import returns the same object from `sys.modules`.

In [1]:
import importlib
import sys
import tempfile
from pathlib import Path

module_name = "demo_mod"
original_sys_path = sys.path.copy()
old_module = sys.modules.get(module_name)
had_old_module = module_name in sys.modules

try:
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp)
        (path / f"{module_name}.py").write_text("VALUE = 100\n", encoding="utf-8")
        sys.path.insert(0, str(path))
        importlib.invalidate_caches()

        first = importlib.import_module(module_name)
        first_id = id(first)

        assert first.VALUE == 100
        assert sys.modules[module_name] is first

        del first

        second = importlib.import_module(module_name)

        assert id(second) == first_id
        assert sys.modules[module_name] is second
        assert second.VALUE == 100
finally:
    sys.path[:] = original_sys_path
    if had_old_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

print("Passed: repeated import reused the cached module object.")

Passed: repeated import reused the cached module object.


## Problem 2 — Rebinding a name is not removing a module

Import `math`, save the module object, rebind the name `math` to `None`, and prove that `sys.modules["math"]` still contains the original module. Then re-import `math` and prove that the name again points to the cached module.

In [2]:
import math
import sys

original_math = math
original_id = id(math)

math = None

assert math is None
assert "math" in sys.modules
assert sys.modules["math"] is original_math
assert id(sys.modules["math"]) == original_id

import math

assert math is original_math
assert id(math) == original_id

print("Passed: namespace rebinding did not affect sys.modules.")

Passed: namespace rebinding did not affect sys.modules.


## Problem 3 — Build and import a runtime-created module

Use `types.ModuleType` to create a module object dynamically. Add a `Point` type and a `distance` function. Register it in `sys.modules`, import it by name, and clean up afterward.

In [3]:
import importlib
import math
import sys
import types
from collections import namedtuple

module_name = "geometry_runtime"
old_module = sys.modules.get(module_name)
had_old_module = module_name in sys.modules

try:
    mod = types.ModuleType(module_name, "Runtime-generated geometry helpers.")
    mod.Point = namedtuple("Point", "x y")

    def distance(p1, p2):
        return math.hypot(p1.x - p2.x, p1.y - p2.y)

    mod.distance = distance
    sys.modules[module_name] = mod

    imported = importlib.import_module(module_name)

    assert imported is mod
    assert imported.__name__ == module_name
    assert imported.__doc__ == "Runtime-generated geometry helpers."
    assert imported.distance(imported.Point(0, 0), imported.Point(3, 4)) == 5.0
finally:
    if had_old_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

print("Passed: dynamic module was importable through sys.modules.")

Passed: dynamic module was importable through sys.modules.


## Problem 4 — Verify that module attributes live in `__dict__`

Create a dynamic module and add attributes in three ways: direct attribute assignment, direct `__dict__` mutation, and `setattr`. Prove that each attribute is stored in the module namespace dictionary.

In [4]:
import types

mod = types.ModuleType("namespace_lab")

mod.answer = 42
mod.__dict__["language"] = "Python"
setattr(mod, "items", [1, 2, 3])

assert mod.answer == mod.__dict__["answer"]
assert mod.language == mod.__dict__["language"]
assert mod.items is mod.__dict__["items"]

mod.items.append(4)
assert mod.__dict__["items"] == [1, 2, 3, 4]

print("Passed: module attributes and module.__dict__ entries are the same namespace.")

Passed: module attributes and module.__dict__ entries are the same namespace.


## Problem 5 — Show when top-level module code executes

Create a module whose top-level code increments a counter stored in `builtins`. Prove that ordinary repeated imports do not re-execute the file, but `importlib.reload` does.

In [5]:
import builtins
import importlib
import sys
import tempfile
from pathlib import Path

module_name = "execution_counter_mod"
counter_name = "_module_execution_counter"
original_sys_path = sys.path.copy()
old_module = sys.modules.get(module_name)
had_old_module = module_name in sys.modules
had_counter = hasattr(builtins, counter_name)
old_counter = getattr(builtins, counter_name, None)

try:
    setattr(builtins, counter_name, 0)

    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp)
        (path / f"{module_name}.py").write_text(
            "import builtins\n"
            f"builtins.{counter_name} += 1\n"
            "VALUE = 'loaded'\n",
            encoding="utf-8"
        )
        sys.path.insert(0, str(path))
        importlib.invalidate_caches()

        m1 = importlib.import_module(module_name)
        assert getattr(builtins, counter_name) == 1

        m2 = importlib.import_module(module_name)
        assert m2 is m1
        assert getattr(builtins, counter_name) == 1

        importlib.reload(m1)
        assert getattr(builtins, counter_name) == 2
finally:
    sys.path[:] = original_sys_path
    if had_old_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

    if had_counter:
        setattr(builtins, counter_name, old_counter)
    else:
        delattr(builtins, counter_name)

print("Passed: repeated imports used the cache; reload re-executed top-level code.")

Passed: repeated imports used the cache; reload re-executed top-level code.


## Problem 6 — Observe partial module initialization during circular imports

Create two temporary modules, `a.py` and `b.py`, that import each other. Record whether `a` exists in `sys.modules` while `b` is being executed. This demonstrates that Python inserts a module into `sys.modules` before its top-level code has finished executing.

In [6]:
import importlib
import sys
import tempfile
from pathlib import Path

modules = ["a", "b"]
old_modules = {name: sys.modules.get(name) for name in modules}
had_modules = {name: name in sys.modules for name in modules}
original_sys_path = sys.path.copy()

try:
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp)
        (path / "a.py").write_text(
            "STATE = ['a started']\n"
            "import b\n"
            "STATE.append('a finished')\n"
            "A_READY = True\n",
            encoding="utf-8"
        )
        (path / "b.py").write_text(
            "import sys\n"
            "A_WAS_IN_CACHE = 'a' in sys.modules\n"
            "A_HAD_READY_FLAG = hasattr(sys.modules.get('a'), 'A_READY')\n"
            "B_READY = True\n",
            encoding="utf-8"
        )

        sys.path.insert(0, str(path))
        importlib.invalidate_caches()

        a = importlib.import_module("a")
        b = importlib.import_module("b")

        assert a.STATE == ["a started", "a finished"]
        assert b.A_WAS_IN_CACHE is True
        assert b.A_HAD_READY_FLAG is False
        assert a.A_READY is True
        assert b.B_READY is True
finally:
    sys.path[:] = original_sys_path
    for name in modules:
        if had_modules[name]:
            sys.modules[name] = old_modules[name]
        else:
            sys.modules.pop(name, None)

print("Passed: circular import exposed a partially initialized module in sys.modules.")

Passed: circular import exposed a partially initialized module in sys.modules.


## Problem 7 — Write a context manager for safe temporary imports

Implement a context manager that temporarily adds a directory to `sys.path`, invalidates import caches, and restores both `sys.path` and selected `sys.modules` entries afterward.

In [7]:
from contextlib import contextmanager
import importlib
import sys
import tempfile
from pathlib import Path

@contextmanager
def temporary_import_path(path, *module_names):
    old_path = sys.path.copy()
    old_modules = {name: sys.modules.get(name) for name in module_names}
    had_modules = {name: name in sys.modules for name in module_names}
    try:
        sys.path.insert(0, str(path))
        importlib.invalidate_caches()
        yield
    finally:
        sys.path[:] = old_path
        for name in module_names:
            if had_modules[name]:
                sys.modules[name] = old_modules[name]
            else:
                sys.modules.pop(name, None)
        importlib.invalidate_caches()

module_name = "settings"

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp)
    (path / f"{module_name}.py").write_text(
        "DEBUG = True\nDATABASE = 'sqlite:///:memory:'\n",
        encoding="utf-8"
    )

    with temporary_import_path(path, module_name):
        settings = importlib.import_module(module_name)
        assert settings.DEBUG is True
        assert settings.DATABASE == "sqlite:///:memory:"
        assert module_name in sys.modules

    assert module_name not in sys.modules

print("Passed: temporary import context manager isolated import state.")

Passed: temporary import context manager isolated import state.


## Problem 8 — Build a mini plugin registry from dynamic modules

Create several runtime-generated plugin modules. Each plugin must expose `PLUGIN_NAME` and a callable `run`. Register them in `sys.modules`, discover modules by prefix, execute them, and restore import state.

In [8]:
import sys
import types

def make_plugin(module_name, plugin_name, transform):
    mod = types.ModuleType(module_name)
    mod.PLUGIN_NAME = plugin_name
    mod.run = transform
    return mod

plugin_specs = {
    "plugin_upper": ("upper", lambda text: text.upper()),
    "plugin_reverse": ("reverse", lambda text: text[::-1]),
    "plugin_title": ("title", lambda text: text.title()),
}

old_modules = {name: sys.modules.get(name) for name in plugin_specs}
had_modules = {name: name in sys.modules for name in plugin_specs}

try:
    for module_name, (plugin_name, transform) in plugin_specs.items():
        sys.modules[module_name] = make_plugin(module_name, plugin_name, transform)

    discovered = {
        name: mod
        for name, mod in sys.modules.items()
        if name.startswith("plugin_") and name in plugin_specs
    }

    assert set(discovered) == set(plugin_specs)

    results = {}
    for module_name, mod in discovered.items():
        assert isinstance(mod.PLUGIN_NAME, str)
        assert callable(mod.run)
        results[mod.PLUGIN_NAME] = mod.run("python modules")

    assert results == {
        "upper": "PYTHON MODULES",
        "reverse": "seludom nohtyp",
        "title": "Python Modules",
    }
finally:
    for name in plugin_specs:
        if had_modules[name]:
            sys.modules[name] = old_modules[name]
        else:
            sys.modules.pop(name, None)

print("Passed: dynamic plugin modules were registered, discovered, and executed.")
print(results)

Passed: dynamic plugin modules were registered, discovered, and executed.
{'upper': 'PYTHON MODULES', 'reverse': 'seludom nohtyp', 'title': 'Python Modules'}


## Problem 9 — Demonstrate safe module shadowing

Create a temporary module with a name that shadows a normal import name. Prove that a newly imported custom module can be shadowed by `sys.path` order, while already-cached modules are usually returned from `sys.modules`.

In [9]:
import importlib
import sys
import tempfile
from pathlib import Path

module_name = "shadow_target"
old_module = sys.modules.get(module_name)
had_old_module = module_name in sys.modules
original_sys_path = sys.path.copy()

try:
    sys.modules.pop(module_name, None)

    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp)
        (path / f"{module_name}.py").write_text(
            "ORIGIN = 'temporary shadow module'\n",
            encoding="utf-8"
        )
        sys.path.insert(0, str(path))
        importlib.invalidate_caches()

        mod = importlib.import_module(module_name)

        assert mod.ORIGIN == "temporary shadow module"
        assert Path(mod.__file__).name == f"{module_name}.py"
        assert sys.modules[module_name] is mod
finally:
    sys.path[:] = original_sys_path
    if had_old_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

print("Passed: sys.path order can shadow import names for modules not already cached.")

Passed: sys.path order can shadow import names for modules not already cached.


## Problem 10 — Design a reusable `sys.modules` preservation helper

Write a context manager that snapshots selected `sys.modules` bindings, allows arbitrary mutations inside the context, and restores the original state even if an exception occurs.

In [10]:
from contextlib import contextmanager
import sys
import types

@contextmanager
def preserved_modules(*module_names):
    snapshot = {name: sys.modules.get(name) for name in module_names}
    had_module = {name: name in sys.modules for name in module_names}
    try:
        yield
    finally:
        for name in module_names:
            if had_module[name]:
                sys.modules[name] = snapshot[name]
            else:
                sys.modules.pop(name, None)

module_name = "temporary_test_double"
assert module_name not in sys.modules

try:
    with preserved_modules(module_name):
        fake = types.ModuleType(module_name)
        fake.VALUE = "fake dependency"
        sys.modules[module_name] = fake
        assert sys.modules[module_name].VALUE == "fake dependency"
        raise RuntimeError("Intentional test error")
except RuntimeError as exc:
    assert str(exc) == "Intentional test error"

assert module_name not in sys.modules

print("Passed: module-state helper restored sys.modules after an exception.")

Passed: module-state helper restored sys.modules after an exception.


# Final Notes

Key takeaways:

1. A module is an object, usually an instance of `types.ModuleType`.
2. A module namespace is stored in the module object's `__dict__`.
3. Importing binds a name in the current namespace, but the module object is cached in `sys.modules`.
4. Rebinding or deleting a local/global name does not necessarily remove a module from the import cache.
5. Dynamic modules become importable when registered in `sys.modules`.
6. Import experiments should restore `sys.path`, `sys.modules`, and any other global state they mutate.